# Отчёт по лабораторной работе №5

## Разработка клиента (клиент-серверное приложение)

**Дисциплина:** Разработка кроссплатформенных приложений, семестр 2

**Выполнил:** студент группы 12002408

Блинов Кирилл Игоревич

**Проверил:** ассистент кафедры информационных и робототехнических систем

Пузырев Никита Сергеевич

<br>
<br>

**Сайт курсового проекта:** [https://hki-stroitel.ru/](https://hki-stroitel.ru/)


<p align="justify"><strong>Цель работы:</strong> получить практические навыки работы с Spring Framework и разработать клиентское приложение для взаимодействия с сервером по REST API.</p>

<h2 align="center">Задание</h2>

1. Изучить теоретический материал.
2. Написать приложение, следуя примеру из методических указаний.
3. Выполнить дополнительное задание.
4. Сделать отчёт.

Дополнительное задание:
- реализовать вывод дополнительных данных из базы данных, добавленных в прошлой работе;
- реализовать функционал удаления задач в БД из клиентского приложения (право на удаление только у определённой роли);
- реализовать функционал добавления задач в БД из клиентского приложения;
- реализовать функционал сортировки задач;
- улучшить пользовательский интерфейс.


## Ход работы

### 1. Клиент-серверная архитектура

За основу взят серверный проект из лабораторной работы №4 на базе Spring Boot, Spring Security, Spring Data JPA и PostgreSQL 16 (в Docker). Сервер запускается на порту `8085` (PostgreSQL на порту `5545`) и предоставляет REST API для работы с задачами подготовки мероприятий.

В проекте реализованы два клиента, работающих с сервером по HTTP и JSON:
- десктопный клиент на Java Swing (`client/LoginForm.java` и `client/TaskManagerForm.java`), разработанный по методическим указаниям с выполнением всех пунктов дополнительного задания;
- веб-клиент на React (`frontend/`), встроенный в итоговый JAR-архив и доступный в браузере по адресу `http://localhost:8085/login`.

Отчёт содержит только текстовые ячейки с описанием архитектуры и примерами Java-кода. Установка зависимостей и запуск приложения в Google Colab не выполняются.


### 2. Серверный REST API и ограничение удаления по роли

В контроллере `TaskController` (маршрут `/api/tasks`) добавлено разграничение прав на удаление задач: просмотр, создание, редактирование и работа с подзадачами доступны ролям `ADMIN` и `METHODIST`, а удаление задачи (`DELETE /api/tasks/{id}`) разрешено только роли `ADMIN` с помощью `@PreAuthorize("hasRole('ADMIN')")`.

| Метод и маршрут | Роли | Назначение |
| --- | --- | --- |
| `GET /api/auth/csrf` | Все | Получение CSRF-токена и заголовка |
| `POST /api/auth/login` | Все | Авторизация и создание сессионной cookie `RASPISA_LAB5_SESSION` |
| `GET /api/tasks` | `ADMIN`, `METHODIST` | Получение списка задач с дополнительными полями из PostgreSQL |
| `POST /api/tasks` | `ADMIN`, `METHODIST` | Создание новой задачи и запись события `CREATE` в историю |
| `PUT /api/tasks/{id}` | `ADMIN`, `METHODIST` | Обновление задачи |
| `DELETE /api/tasks/{id}` | Только `ADMIN` | Мягкое удаление задачи и запись события `DELETE` в историю |
| `GET /api/tasks/{id}/history` | `ADMIN`, `METHODIST` | Получение истории изменений задачи |


```java
package ru.raspisa.web;

import jakarta.validation.Valid;
import org.springframework.http.HttpStatus;
import org.springframework.http.ResponseEntity;
import org.springframework.security.access.prepost.PreAuthorize;
import org.springframework.web.bind.annotation.*;
import ru.raspisa.service.TaskService;
import ru.raspisa.service.TaskService.*;
import java.util.List;

@RestController
@RequestMapping("/api/tasks")
@PreAuthorize("hasAnyRole('ADMIN', 'METHODIST')")
public class TaskController {
    private final TaskService tasks;
    public TaskController(TaskService tasks) { this.tasks = tasks; }

    @GetMapping
    public List<TaskView> list() { return tasks.list(); }

    @PostMapping
    public ResponseEntity<TaskView> create(@Valid @RequestBody TaskRequest request) {
        return ResponseEntity.status(HttpStatus.CREATED).body(tasks.create(request));
    }

    @PutMapping("/{id}")
    public TaskView update(@PathVariable long id, @Valid @RequestBody TaskRequest request) {
        return tasks.update(id, request);
    }

    @DeleteMapping("/{id}")
    @PreAuthorize("hasRole('ADMIN')")
    @ResponseStatus(HttpStatus.NO_CONTENT)
    public void delete(@PathVariable long id) { tasks.delete(id); }

    @GetMapping("/{id}/history")
    public List<HistoryView> history(@PathVariable long id) { return tasks.history(id); }
}
```


### 3. Форма авторизации десктопного клиента (`LoginForm.java`)

Класс `LoginForm` наследуется от `JFrame`. При нажатии кнопки «Войти» клиент включает хранение cookie (`CookieManager`), запрашивает CSRF-токен по адресу `/api/auth/csrf`, отправляет `POST /api/auth/login` с логином и паролем в формате JSON и при статусе `200` открывает главное окно `TaskManagerForm`, передавая в него имя пользователя и роль.


```java
import javax.swing.*;
import java.awt.*;
import java.net.CookieHandler;
import java.net.CookieManager;
import java.net.CookiePolicy;

public class LoginForm extends JFrame {
    private static final String BASE_URL = "http://localhost:8085";
    private final JTextField usernameField = new JTextField("admin");
    private final JPasswordField passwordField = new JPasswordField("Admin!2026");
    private final JButton loginButton = new JButton("Войти");

    public LoginForm() {
        setTitle("Вход – Raspisa Client");
        setSize(340, 170);
        setLocationRelativeTo(null);
        setDefaultCloseOperation(JFrame.EXIT_ON_CLOSE);

        JPanel panel = new JPanel(new GridLayout(3, 2, 8, 8));
        panel.setBorder(BorderFactory.createEmptyBorder(14, 14, 14, 14));
        panel.add(new JLabel("Логин:"));
        panel.add(usernameField);
        panel.add(new JLabel("Пароль:"));
        panel.add(passwordField);
        panel.add(new JLabel());
        panel.add(loginButton);
        add(panel);

        loginButton.addActionListener(e -> login());
        getRootPane().setDefaultButton(loginButton);
    }

    private void login() {
        String username = usernameField.getText().trim();
        String password = new String(passwordField.getPassword());
        try {
            CookieHandler.setDefault(new CookieManager(null, CookiePolicy.ACCEPT_ALL));
            String[] csrf = TaskManagerForm.fetchCsrf(BASE_URL);
            String body = "{\"login\":\"" + TaskManagerForm.escapeJson(username)
                    + "\",\"password\":\"" + TaskManagerForm.escapeJson(password) + "\"}";
            TaskManagerForm.HttpResult res = TaskManagerForm.sendRequest(
                    BASE_URL + "/api/auth/login", "POST", body, csrf[0], csrf[1]);

            if (res.status == 200) {
                String role = TaskManagerForm.extractString(res.body, "role");
                new TaskManagerForm(BASE_URL, username, role != null ? role : "").setVisible(true);
                dispose();
            } else {
                JOptionPane.showMessageDialog(this, "Неверный логин или пароль (" + res.status + ")");
            }
        } catch (Exception ex) {
            JOptionPane.showMessageDialog(this, "Нет связи с сервером: " + ex.getMessage());
        }
    }

    public static void main(String[] args) {
        SwingUtilities.invokeLater(() -> new LoginForm().setVisible(true));
    }
}
```


### 4. Главное окно клиента (`TaskManagerForm.java`) и дополнительное задание

В классе `TaskManagerForm` реализованы все требования дополнительного задания:
1. **Вывод дополнительных данных из БД:** вместо простого списка `JList` используется таблица `JTable` со столбцами `ID`, `Название`, `Описание`, `Категория`, `Приоритет`, `Статус`, `Автор` и `Создана`.
2. **Добавление задач:** кнопка «Добавить задачу» открывает диалог ввода всех полей и отправляет `POST /api/tasks` с CSRF-токеном.
3. **Удаление задач с проверкой роли:** кнопка «Удалить задачу» активна только для роли `ADMIN` и вызывает `DELETE /api/tasks/{id}`; дополнительно право проверяется на сервере через `@PreAuthorize("hasRole('ADMIN')")`.
4. **Сортировка задач:** в верхней панели расположен выпадающий список с вариантами сортировки по дате создания, приоритету, статусу и названию.
5. **Улучшенный интерфейс:** использованы таблица с прокруткой, отступы, диалоговые окна подтверждения и вывод имени пользователя и роли в заголовке окна.


```java
private void renderTable() {
    List<TaskItem> sorted = new ArrayList<>(items);
    int mode = sortBox.getSelectedIndex();
    if (mode == 0) {
        sorted.sort(Comparator.comparing((TaskItem t) -> t.createdAt).reversed());
    } else if (mode == 1) {
        sorted.sort(Comparator.comparingInt((TaskItem t) -> priorityRank(t.priority)).reversed());
    } else if (mode == 2) {
        sorted.sort(Comparator.comparing((TaskItem t) -> t.status));
    } else if (mode == 3) {
        sorted.sort(Comparator.comparing((TaskItem t) -> t.title, String.CASE_INSENSITIVE_ORDER));
    }
    tableModel.setRowCount(0);
    for (TaskItem t : sorted) {
        tableModel.addRow(new Object[]{
                t.id, t.title, t.description, t.category,
                t.priority, t.status, t.owner, t.createdAt
        });
    }
}

private void deleteSelectedTask() {
    int row = table.getSelectedRow();
    if (row < 0) return;
    long id = Long.parseLong(tableModel.getValueAt(row, 0).toString());
    try {
        String[] csrf = fetchCsrf(baseUrl);
        HttpResult res = sendRequest(baseUrl + "/api/tasks/" + id, "DELETE", null, csrf[0], csrf[1]);
        if (res.status == 204 || res.status == 200) loadTasks();
    } catch (Exception ex) {
        JOptionPane.showMessageDialog(this, "Ошибка сети: " + ex.getMessage());
    }
}
```


### 5. Учётные записи и запуск

| Логин | Пароль | Роль | Доступ к задачам |
| --- | --- | --- | --- |
| `admin` | `Admin!2026` | `ADMIN` | Просмотр, добавление, редактирование и удаление задач |
| `methodist` | `Methodist!2026` | `METHODIST` | Просмотр, добавление и редактирование задач (удаление запрещено) |
| `teacher` | `Teacher!2026` | `TEACHER` | Доступ к журналу посещаемости |
| `student` | `Student!2026` | `STUDENT` | Только просмотр расписания и мероприятий |

Веб-клиент доступен по адресу `http://localhost:8085/login`, а десктопный Swing-клиент запускается из папки `client` командами `javac -encoding UTF-8 LoginForm.java TaskManagerForm.java` и `java LoginForm`.


## Вывод

Получил практические навыки работы с Spring Framework и разработал клиентское приложение для взаимодействия с сервером по REST API.
